| [![back](../../media/navigation/previous.png)](../../exercises/chapter-03-02/ex02.html) | [![home](../../media/navigation/home.png)](../../index.html) | [![next](../../media/navigation/next.png)](../../exercises/chapter-03-02/ex04.html) |
| :---: | :---: | :---: |
| Ex 3.2.2: Count DNA damage foci in nuclei | • | Ex 3.2.4: Call Cellpose from a notebook |

# 3.2 CellPose

## 3.2.3 Evaluate nuclear-cytoplasmic translocation

#### Required data:

| **Folder** | Description | Location | License |
| --- | --- | --- | --- |
| S-BIAD478 | Fluo cells (2D) at three time points | [EMBL-EBI](https://www.ebi.ac.uk/biostudies/studies/S-BIAD478) | CC-BY-4.0 |

#### Goals:

In this part, we will work on images having a nuclear staining (DAPI) and a signal migrating from cytoplasm to nuclei.
We will try to:
- segment both the nucleus and the cytoplasm of each cell
- bind each nucleus to its cell
- discard cells without nucleus and vice-versa
- discard cells cut by the border
- measure the translocation factor
We will do that in QuPath with Cellpose.
The process will be as automated as possible so it can be applied to a whole folder of images in one shot.


<center>
<img src="../../media/chapter03_02/cp-qupath-intro.png" alt="segmented cells" width="75%" >
<br>   
<small>
Fig. 1.1 &ndash; Segmented cells in QuPath.
</small>
</center>

### 3.2.3.1 Quantify nuclear-cytoplasmic translocation in QuPath

#### A. Goals

- QuPath is a very efficient software when it comes to process big amounts of 2D data, whether it is massive slides (using pyramids) or important quantity of images (batch processing).
- Cellpose can be used from QuPath through a Groovy script (QuPath scripting language).


#### B. Setup a project

- To work on more than one image, QuPath requires to create a project. You can do that by going to the top-bar menu, in "File" > "New project". At this point, you must provide an **empty folder**.
- Once your project is created, take all the images from the first time point (at 0 minute) and import them into QuPath (the drag-and-drop works).
- QuPath will ask you what these images are, simply set it to "Fluorescence".
- Save your project with [Ctrl+S] or "File" > "Save".


<center>
<img src="../../media/chapter03_02/import-qupath.png" alt="import images in qp" width="20%" >
<br>   
<small>
Fig. 1.2 &ndash; Importing a batch of images.
</small>
</center>

**Note:**

QuPath crashes quite often, take the habit to save regularly when using this application.

#### C. Segment the cells (nuclei and cytoplasm)

- Start by opening any image you want by double-cliking on it in the left column on QuPath.
- Then, in the top-bar menu, you can go to "Extensions" > "Cellpose" > "Detect nuclei and cells using Cellpose".
- The script editor should pop-up with a template script doing precisely what we want. We just need to adjust the settings.

##### a. Create a full-image annotation

- As mentionned earlier, QuPath can open whole slide images but it doesn't mean that we want to work the whole image all the time.
- For this reason, the majority of QuPath's operations only work if you provide an annotation (== polygon) to work in.
- We want to automate the process as much as possible, so we don't want to manually open each individual image and create an annotation.
- To do that automatically, at the begining of the script, we will add a piece of code to:
    - Select present cell detections (potential previous attempts) and delete them
    - Select present annotations (potential previous attempts) and delete them
    - Create a fresh new full-image annotation.
- You can add to your script the following snippet of code:

``` groovy
selectDetections()
removeSelectedObjects()
selectAnnotations()
removeSelectedObjects()
createFullImageAnnotation(true) // the 'true' means that the annotation will be selected
```

##### b. Load the segmentation script template

- If you scroll in the code, you will notice two very similar lines looking like `def cellpose_xxx = Cellpose2D.builder(...`.
- The first one and the lines starting with a dot after it are responsible for the cytoplasm segmentation, while the other is responsible for the nuclei segmentation.
- All the remaining code is just to bind polygons together (a polygon of cyto + a polygon of nucleus) and discard nucleus-less cytoplasms and vice-versa.
- The first thing that we can start to do is to update the models to use. The script contains `cyto3` for both objects, but this model doesn't exist anymore for Cellpose 4. Instead you have to use `cpsam` (that happens to be the only available default model in Cellpose 4).
- Then we can change the channel(s) to look at. This takes place in the `.channels(xxx)` instruction. If you open the ◑ brightness and contrast tool, you can see the name of channels in there. The channels should be simply named "Red" and "Green".
- The cytoplasm is contained in the "Green" channel so you "HCS" by "Green". Of the same way, you can replace "DAPI" by "Red".
- You certainly noticed that "DAPI" is repeated twice: once after the cytoplasm's channel name, and once by itself. This is because Cellpose can use a nuclear channel when segmenting cytoplasm to help in finding the boundary between two cells.

##### c. 'Pixel size' or 'diameter'?

- The most important thing that we need to address right away is the use of ".pixelSize" vs. ".diameter" that you can see in the list of settings.
    - **pixelSize** determines the downsampling used on the image that we will feed to CellPose. The bigger the pixel size is, the smaller the objects are if you measure diameters in number of pixels.
    - **diameter** is the median diameter (in number of pixels) of the objects that you want to segment. CellPose will automatically rescale your image so the length that you provided here becomes 30 pixels *(ex: if your image contains a circle of 45 pixels of diameter and you provide 45 in the field "diameter", CellPose will rescale the image for the circle to be 30 pixels of diameter.)*.
- The problem comes from the fact that CellPose expects the objects to have a median diameter of 30 pixels. So:
    - If you change the pixel size, the diameter in pixels that you observe on your image won't be correct, it won't match the downsampled image.
    - If you change the diameter value, it will invalidate the downsampling that you provided.
- Actually, both these settings influence the same thing: they both result in a resampling of the image. If you use them both, you will do a double-resampling which is very hard to manage. You should either:
    - keep **diameter** at 30 and play with the pixel size.
    - set the **pixel size** to the original pixel size and tune the diameter.
- We will go with the second option in this exercise, simply because it is easier to roughly eyeball a number of pixels rather than calculating by which downsampling factor our objects will end up being 30 pixels in diameter.
- If you go in the "Image" tab (next to "Project", "Annotations", ...) and check the pixel size, you will notice that these images are not calibrated (both their pixel height and width are at 1 or empty). 
- You should then give 1.0 to both ".pixelSize" calls in the script.

##### d. Additional settings

- For consistent results, we need to normalize the intensity of the images that we feed to Cellpose, but not the original images.
- To do so, you need to add the following line in both configurations.

```groovy
.normalizePercentilesGlobal(0.1, 99.8, 5)
```

- If you have troubles editing the script, you can find [a clean version here](./cp-qupath-clean.html).
- You can now click on the three vertical dots next to "Run" in the lower-right corner and use "Run for project".
- Move all images in the right column and launch the process.

#### D. Measure the translocation ratio

- The translocation consists in observing the signal moving from the cytoplasm to the nucleus or vice-versa.
- We will process the translocation ratio for every individual cell using the following formula:

$$
r = \frac{I_{nuc} - I_{cyto}}{I_{nuc} + I_{cyto}}
$$

$$
I_{nuc} = \max\left(0,\ \tilde{x}_{nuc} + 1.28 \cdot \sigma_{nuc}\right)
$$

$$
I_{cyto} = \max\left(0,\ \tilde{x}_{cyto} + 1.28 \cdot \sigma_{cyto}\right)
$$

Where:
- $r$ : translocation ratio, bounded in $[-1, 1]$
- $\tilde{x}_{nuc}$ : median intensity in the nucleus
- $\tilde{x}_{cyto}$ : median intensity in the cytoplasm
- $\sigma_{nuc}$ : standard deviation of intensity in the nucleus
- $\sigma_{cyto}$ : standard deviation of intensity in the cytoplasm
- $1.28$ : z-score corresponding to the 90th quantile under a normal distribution assumption

- If you click on any cell and go to the "Annotations" tab of QuPath, you can observe in the lower-left corner that there are metrics such as:
    - "Cytoplasm: Green: Median"
    - "Cytoplasm: Green: Std.Dev."
    - ...
- We will use these from a script to process the translocation ratio and inject it back into the cells.
- You can find here the [translocation measurement script](./translocation-measurement.html).

#### E. Export measurements

- Once all the script are done, save your project with [Ctrl+S].
- Then, in QuPath's top-bar menu, go to "Measure" > "Export measurements...".
- Move all the images to the right column.
- Select a location where the TSV file containing measurements will be saved. The default proposed location is fine.
- Set the "Export type" to "Cells".
- Click on "Populate" and, in the dropdown menu, select "Image" and "Translocation ratio".
- If you click on "Export", a TSV file should have been created.

**Note:**

If you don't select anything in the populated list, everything will be exported.

#### F. Plot the results

- In the data associated to the exercise, you will find QuPath projects for the images at 20 minutes and 40 minutes. Both these projects contain a file "measurements.tsv" that have the same shape as what you just created.
- Once you exported your measurements for 00 minutes and found the two other measurement files, open with Jupyter the notebook `notebooks/chapter-03-02/plot-translocation.ipynb`.
- The instructions to plot the results are in the notebook itself.
- Given the fact that the translocation is supposed to occur from the cytoplasm to the nucleus, if the experiment worked, we should see values shifting from negative values to positive values.